In [1]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import time

In [2]:
# Load dataset
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Check dataset shape
print("Training set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

# Display first few rows
train_df.head()

Training set shape: (8693, 14)
Test set shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
# Apply unified preprocessing strategy

# Make copies to avoid changing the original data
train_processed = train_df.copy()
test_processed = test_df.copy()

# Save PassengerId for later Kaggle submission
test_passenger_id = test_processed["PassengerId"]

# Separate target variable
y = train_processed["Transported"].astype(int)
train_processed = train_processed.drop(columns=["Transported"])

# Define feature groups
monetary_features = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
categorical_features = ["HomePlanet", "Destination", "CryoSleep", "VIP"]

# 1. Monetary features: fill missing values with 0
for col in monetary_features:
    train_processed[col] = train_processed[col].fillna(0)
    test_processed[col] = test_processed[col].fillna(0)

# 2. Apply log1p transformation to reduce right-skewness
for col in monetary_features:
    train_processed[col] = np.log1p(train_processed[col])
    test_processed[col] = np.log1p(test_processed[col])

# 3. Categorical variables: fill missing values with mode from training set
# Use option_context and infer_objects to avoid pandas FutureWarning
for col in categorical_features:
    mode_value = train_processed[col].mode()[0]
    
    with pd.option_context("future.no_silent_downcasting", True):
        train_processed[col] = train_processed[col].fillna(mode_value).infer_objects(copy=False)
        test_processed[col] = test_processed[col].fillna(mode_value).infer_objects(copy=False)

# 4. Age: fill missing values with median from training set
age_median = train_processed["Age"].median()
train_processed["Age"] = train_processed["Age"].fillna(age_median)
test_processed["Age"] = test_processed["Age"].fillna(age_median)

# 5. Bin Age into four ordinal categories
# 0 = child, 1 = young, 2 = adult, 3 = senior
age_bins = [-1, 12, 25, 60, np.inf]
age_labels = [0, 1, 2, 3]

train_processed["AgeGroup"] = pd.cut(
    train_processed["Age"],
    bins=age_bins,
    labels=age_labels
).astype(int)

test_processed["AgeGroup"] = pd.cut(
    test_processed["Age"],
    bins=age_bins,
    labels=age_labels
).astype(int)

# Drop original Age after binning
train_processed = train_processed.drop(columns=["Age"])
test_processed = test_processed.drop(columns=["Age"])

# 6. Drop irrelevant features
drop_features = ["Name", "PassengerId", "Cabin"]

train_processed = train_processed.drop(columns=drop_features)
test_processed = test_processed.drop(columns=drop_features)

# Final feature matrices
X = train_processed
X_test = test_processed

# Check processed data
print("Processed training features shape:", X.shape)
print("Processed test features shape:", X_test.shape)
print("Target shape:", y.shape)

print("\nMissing values in processed training data:")
print(X.isnull().sum())

X.head()

Processed training features shape: (8693, 10)
Processed test features shape: (4277, 10)
Target shape: (8693,)

Missing values in processed training data:
HomePlanet      0
CryoSleep       0
Destination     0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
AgeGroup        0
dtype: int64


,HomePlanet,CryoSleep,Destination,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,AgeGroup
0,Europa,False,TRAPPIST-1e,False,0.000000,0.000000,0.000000,0.000000,0.000000,2
1,Earth,False,TRAPPIST-1e,False,4.700480,2.302585,3.258097,6.309918,3.806662,1
2,Europa,False,TRAPPIST-1e,True,3.784190,8.182280,0.000000,8.812248,3.912023,2
3,Europa,False,TRAPPIST-1e,False,0.000000,7.157735,5.918894,8.110728,5.267858,2
4,Earth,False,TRAPPIST-1e,False,5.717028,4.262680,5.023881,6.338594,1.098612,1


In [5]:
# SVM-specific encoding, scaling, and train-validation split

# Define numerical and categorical features after preprocessing
numeric_features = [
    "RoomService", 
    "FoodCourt", 
    "ShoppingMall", 
    "Spa", 
    "VRDeck", 
    "AgeGroup"
]

categorical_features = [
    "HomePlanet", 
    "CryoSleep", 
    "Destination", 
    "VIP"
]

# For SVM:
# - numerical features need scaling
# - categorical features need one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Split the training data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

print("\nTarget distribution in training set:")
print(y_train.value_counts(normalize=True))

print("\nTarget distribution in validation set:")
print(y_valid.value_counts(normalize=True))

X_train shape: (6954, 10)
X_valid shape: (1739, 10)
y_train shape: (6954,)
y_valid shape: (1739,)

Target distribution in training set:
Transported
1    0.503595
0    0.496405
Name: proportion, dtype: float64

Target distribution in validation set:
Transported
1    0.503738
0    0.496262
Name: proportion, dtype: float64


In [6]:
# Train a baseline SVM model

baseline_svm = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(kernel="rbf", C=1.0, gamma="scale"))
    ]
)

# Record training time
start_time = time.time()

# Train the model
baseline_svm.fit(X_train, y_train)

training_time = time.time() - start_time

# Predict on validation set
y_pred = baseline_svm.predict(X_valid)

# Evaluate the model
baseline_accuracy = accuracy_score(y_valid, y_pred)

print("Baseline SVM Validation Accuracy:", baseline_accuracy)
print("Training Time:", training_time, "seconds")

print("\nClassification Report:")
print(classification_report(y_valid, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, y_pred))

Baseline SVM Validation Accuracy: 0.7901092581943646
Training Time: 2.3691840171813965 seconds

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.77      0.78       863
           1       0.78      0.81      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739


Confusion Matrix:
[[661 202]
 [163 713]]


In [8]:
%pip install ipywidgets tqdm

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 6.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 18.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [widgetsnbextension]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Hyperparameter tuning for SVM using Optuna

# Import Optuna. If it is not installed, install it first.
try:
    import optuna
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
    import optuna

# Make Optuna output cleaner
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Objective function for Optuna.
    Each trial trains an RBF-kernel SVM and evaluates it using 5-fold stratified cross-validation.
    """
    
    C = trial.suggest_float("C", 0.1, 100, log=True)
    gamma = trial.suggest_float("gamma", 0.001, 1, log=True)
    
    svm_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", SVC(kernel="rbf", C=C, gamma=gamma))
        ]
    )
    
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    
    scores = cross_val_score(
        svm_model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    
    return scores.mean()

# Run Bayesian optimization
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

start_time = time.time()

study.optimize(objective, n_trials=20)

tuning_time = time.time() - start_time

print("Best parameters:", study.best_params)
print("Best 5-fold CV accuracy:", study.best_value)
print("Tuning time:", tuning_time, "seconds")

Best parameters: {'C': 81.4470878579612, 'gamma': 0.026301587628514228}
Best 5-fold CV accuracy: 0.7953698234798214
Tuning time: 129.30471277236938 seconds


In [10]:
# Train and evaluate the best SVM model

best_params = study.best_params

best_svm = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=best_params["C"],
            gamma=best_params["gamma"]
        ))
    ]
)

# Train the best SVM model
start_time = time.time()

best_svm.fit(X_train, y_train)

best_training_time = time.time() - start_time

# Predict on validation set
y_valid_pred = best_svm.predict(X_valid)

# Evaluate the model
best_valid_accuracy = accuracy_score(y_valid, y_valid_pred)

print("Best SVM Validation Accuracy:", best_valid_accuracy)
print("Best SVM Training Time:", best_training_time, "seconds")

print("\nBest Parameters:")
print(best_params)

print("\nClassification Report:")
print(classification_report(y_valid, y_valid_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, y_valid_pred))

Best SVM Validation Accuracy: 0.7941345600920069
Best SVM Training Time: 7.379770517349243 seconds

Best Parameters:
{'C': 81.4470878579612, 'gamma': 0.026301587628514228}

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.77      0.79       863
           1       0.78      0.82      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739


Confusion Matrix:
[[662 201]
 [157 719]]


In [11]:
# Summarize SVM results

svm_summary = pd.DataFrame({
    "Model": ["Baseline SVM", "Tuned SVM"],
    "Kernel": ["RBF", "RBF"],
    "C": [1.0, best_params["C"]],
    "Gamma": ["scale", best_params["gamma"]],
    "Validation Accuracy": [baseline_accuracy, best_valid_accuracy],
    "5-fold CV Accuracy": [None, study.best_value],
    "Training Time (seconds)": [training_time, best_training_time]
})

svm_summary

,Model,Kernel,C,Gamma,Validation Accuracy,5-fold CV Accuracy,Training Time (seconds)
0,Baseline SVM,RBF,1.000000,scale,0.790109,NaN,2.369184
1,Tuned SVM,RBF,81.447088,0.026302,0.794135,0.79537,7.379771


In [12]:
# Train final SVM model on the full training set and generate Kaggle submission

final_svm = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=best_params["C"],
            gamma=best_params["gamma"]
        ))
    ]
)

# Train on all available training data
final_svm.fit(X, y)

# Predict on the Kaggle test set
test_predictions = final_svm.predict(X_test)

# Create submission file
svm_submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Transported": test_predictions.astype(bool)
})

# Save submission file
svm_submission.to_csv("svm_submission.csv", index=False)

print("Submission file created: svm_submission.csv")
print(svm_submission.head())
print(svm_submission.shape)

Submission file created: svm_submission.csv
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
(4277, 2)


In [13]:
# Local Optuna search around the current best SVM parameters

def objective_local(trial):
    C = trial.suggest_float("C", 20, 200, log=True)
    gamma = trial.suggest_float("gamma", 0.005, 0.08, log=True)
    
    svm_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", SVC(kernel="rbf", C=C, gamma=gamma))
        ]
    )
    
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    
    scores = cross_val_score(
        svm_model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    
    return scores.mean()

local_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=2026)
)

start_time = time.time()

local_study.optimize(objective_local, n_trials=20)

local_tuning_time = time.time() - start_time

print("Local search best parameters:", local_study.best_params)
print("Local search best 5-fold CV accuracy:", local_study.best_value)
print("Local search tuning time:", local_tuning_time, "seconds")

Local search best parameters: {'C': 23.69701231141509, 'gamma': 0.02440313595719461}
Local search best 5-fold CV accuracy: 0.7952264557872034
Local search tuning time: 142.650563955307 seconds
